# A02 - Data Cleaning and Feature Engineering

## Objective
Prepare a consistent dataset for hotel booking cancellation prediction.

## Main Tasks
1. Load the original dataset
2. Create a working copy
3. Remove leakage and unsuitable features
4. Handle missing and invalid values
5. Create new features
6. Export the cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

notebook_path = os.environ.get("JPY_SESSION_NAME")

if notebook_path:
    project_root = Path(notebook_path).resolve().parent.parent
else:
    project_root = Path.cwd().resolve()
    if project_root.name == "notebooks":
        project_root = project_root.parent

csv_path = project_root / "data" / "hotel_bookings.csv"

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("CSV found:", csv_path.exists())

In [ ]:
df_clean = df.copy()

print("Original dataset:", df.shape)
print("Working dataset:", df_clean.shape)

In [ ]:
columns_to_drop = [
    "company",
    "reservation_status",
    "reservation_status_date",
    "booking_changes",
    "assigned_room_type",
    "days_in_waiting_list"
]

df_clean = df_clean.drop(columns=columns_to_drop)

print("Remaining shape:", df_clean.shape)
print("Dropped columns:", columns_to_drop)

In [ ]:
missing_after_drop = df_clean.isna().sum()

print(missing_after_drop[missing_after_drop > 0])

In [ ]:
total_guests = (
    df_clean["adults"]
    + df_clean["children"].fillna(0)
    + df_clean["babies"]
)

total_nights = (
    df_clean["stays_in_week_nights"]
    + df_clean["stays_in_weekend_nights"]
)

print("Negative ADR:", (df_clean["adr"] < 0).sum())
print("Zero total guests:", (total_guests == 0).sum())
print("Zero total nights:", (total_nights == 0).sum())

In [ ]:
print("Target exists:", "is_canceled" in df_clean.columns)
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

print(df_clean["is_canceled"].value_counts())

In [ ]:
# Check missing values before cleaning
print("Before:", df_clean["children"].isna().sum())

# Fill missing children values with 0
df_clean["children"] = df_clean["children"].fillna(0)

# Check missing values after cleaning
print("After:", df_clean["children"].isna().sum())

In [ ]:
# Check missing values before cleaning
print("Before:", df_clean["country"].isna().sum())

# Fill missing country values with Unknown
df_clean["country"] = df_clean["country"].fillna("Unknown")

# Check missing values after cleaning
print("After:", df_clean["country"].isna().sum())

In [ ]:
# Check missing values before cleaning
print("Before:", df_clean["agent"].isna().sum())

# Convert agent IDs to categories
df_clean["agent"] = (
    df_clean["agent"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

# Check missing values after cleaning
print("After:", df_clean["agent"].isna().sum())

# Display some agent categories
print(df_clean["agent"].value_counts().head())

In [ ]:
# Check remaining missing values
remaining_missing = df_clean.isna().sum()

print(remaining_missing[remaining_missing > 0])

print("Dataset shape:", df_clean.shape)

In [ ]:
# Identify abnormal records

negative_adr = df_clean["adr"] < 0

zero_guests = (
    df_clean["adults"]
    + df_clean["children"]
    + df_clean["babies"]
) == 0

zero_nights = (
    df_clean["stays_in_week_nights"]
    + df_clean["stays_in_weekend_nights"]
) == 0

# Count abnormal records
print("Negative ADR:", negative_adr.sum())
print("Zero guests:", zero_guests.sum())
print("Zero nights:", zero_nights.sum())

# Count unique records with at least one issue
any_issue = negative_adr | zero_guests | zero_nights

print("Records with any issue:", any_issue.sum())

In [ ]:
# Record dataset size before cleaning
rows_before = len(df_clean)

# Keep records without the selected issues
df_clean = df_clean.loc[~any_issue].copy()

# Record dataset size after cleaning
rows_after = len(df_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)

In [ ]:
# Check cleaned dataset

print("Final dataset shape:", df_clean.shape)

print("Missing values:", df_clean.isna().sum().sum())

print("Negative ADR:", (df_clean["adr"] < 0).sum())

print(
    "Zero guests:",
    (
        df_clean["adults"]
        + df_clean["children"]
        + df_clean["babies"]
        == 0
    ).sum()
)

print(
    "Zero nights:",
    (
        df_clean["stays_in_week_nights"]
        + df_clean["stays_in_weekend_nights"]
        == 0
    ).sum()
)

print("Target exists:", "is_canceled" in df_clean.columns)

print(
    "Cancellation rate:",
    round(df_clean["is_canceled"].mean() * 100, 2),
    "%"
)

In [ ]:
# Create total number of guests
df_clean["total_guests"] = (
    df_clean["adults"]
    + df_clean["children"]
    + df_clean["babies"]
)

# Create total number of nights
df_clean["total_nights"] = (
    df_clean["stays_in_week_nights"]
    + df_clean["stays_in_weekend_nights"]
)

# Check new features
print(df_clean[[
    "adults",
    "children",
    "babies",
    "total_guests",
    "total_nights"
]].head())

In [ ]:
# Create family booking indicator
df_clean["family_booking"] = (
    (df_clean["children"] > 0)
    | (df_clean["babies"] > 0)
).astype(int)

# Check the distribution
print(df_clean["family_booking"].value_counts())

In [ ]:
# Group bookings by lead time
df_clean["lead_time_group"] = pd.cut(
    df_clean["lead_time"],
    bins=[-1, 7, 30, 90, 180, float("inf")],
    labels=[
        "0-7 days",
        "8-30 days",
        "31-90 days",
        "91-180 days",
        "181+ days"
    ]
)

# Check the distribution
print(df_clean["lead_time_group"].value_counts().sort_index())

In [ ]:
# Check dataset after feature engineering
print("Dataset shape:", df_clean.shape)

# Check missing values
print("Missing values:", df_clean.isna().sum().sum())

# Check new feature names
new_features = [
    "total_guests",
    "total_nights",
    "family_booking",
    "lead_time_group"
]

print("New features exist:")
print(df_clean[new_features].head())

# Check invalid values
print("Zero guests:", (df_clean["total_guests"] == 0).sum())
print("Zero nights:", (df_clean["total_nights"] == 0).sum())

In [ ]:
# Convert total guests to integer
df_clean["total_guests"] = df_clean["total_guests"].astype(int)

# Check data types
print(df_clean[[
    "total_guests",
    "total_nights",
    "family_booking",
    "lead_time_group"
]].dtypes)

In [ ]:
# Step 1: Create arrival season feature

# Define month-to-season mapping
season_map = {
    "January": "Winter",
    "February": "Winter",
    "March": "Spring",
    "April": "Spring",
    "May": "Spring",
    "June": "Summer",
    "July": "Summer",
    "August": "Summer",
    "September": "Autumn",
    "October": "Autumn",
    "November": "Autumn",
    "December": "Winter"
}

# Create a new feature
df_clean["arrival_season"] = (
    df_clean["arrival_date_month"].map(season_map)
)

# Check season distribution
print(df_clean["arrival_season"].value_counts())

# Check missing values
print(
    "Missing season values:",
    df_clean["arrival_season"].isna().sum()
)

# Check dataset shape
print("Dataset shape:", df_clean.shape)

In [ ]:
# Create output folder
clean_data_dir = project_root / "data"
clean_data_dir.mkdir(parents=True, exist_ok=True)

# Define output file path
clean_csv_path = clean_data_dir / "hotel_bookings_clean.csv"

# Save cleaned dataset
df_clean.to_csv(clean_csv_path, index=True, index_label="row_id")

print("File saved to:", clean_csv_path)
print("Final dataset shape:", df_clean.shape)

In [ ]:
# Read the saved CSV file
df_check = pd.read_csv(clean_csv_path)

# Verify the exported dataset
print("Loaded dataset shape:", df_check.shape)
print("Missing values:", df_check.isna().sum().sum())
print("Row ID exists:", "row_id" in df_check.columns)
print("Target exists:", "is_canceled" in df_check.columns)

df_check.head()

In [ ]:
# Verify the updated CSV file
df_check = pd.read_csv(clean_csv_path)

print("Dataset shape:", df_check.shape)
print("Arrival season exists:", "arrival_season" in df_check.columns)
print("Missing values:", df_check.isna().sum().sum())
print("Row IDs unique:", df_check["row_id"].is_unique)